In [0]:
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT :catalog_use;
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT :schema_use;

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog(), current_schema();

In [0]:
%skip
DROP TABLE IF EXISTS fhir_bundle_zerobus;

In [0]:
CREATE TABLE IF NOT EXISTS fhir_bundle_zerobus
(
  bundle_uuid STRING NOT NULL COMMENT 'Unique identifier for the FHIR bundle (primary key)',
  ingest_datetime TIMESTAMP NOT NULL DEFAULT current_timestamp() COMMENT 'Timestamp when the record was ingested into the table',
  fhir VARIANT COMMENT 'FHIR bundle payload stored as VARIANT for flexible JSON querying',
  source_system STRING COMMENT 'Source system identifier for the FHIR message',
  event_timestamp TIMESTAMP COMMENT 'Original event timestamp from the source system',
  user_email STRING COMMENT 'User email address associated with the FHIR message',
  CONSTRAINT fhir_zerobus_pk PRIMARY KEY (bundle_uuid)
)
USING DELTA
CLUSTER BY AUTO
COMMENT 'Target table for zerobus FHIR bundle streaming data with VARIANT JSON storage'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.feature.allowColumnDefaults' = 'supported',
  'delta.feature.variantType-preview' = 'supported',
  'delta.minReaderVersion' = '3',
  'delta.minWriterVersion' = '7',
  'delta.columnMapping.mode' = 'name',
  'quality' = 'bronze',
  'pipeline' = 'fhir_zerobus',
  'description' = 'Zerobus streaming target table for FHIR bundles'
);

In [0]:
SHOW CREATE TABLE fhir_bundle_zerobus;

In [0]:
DECLARE OR REPLACE VARIABLE spn_application_id STRING;

SET VARIABLE spn_application_id = :spn_application_id;

SELECT spn_application_id;

In [0]:
DECLARE OR REPLACE grnt_stmnt STRING DEFAULT "GRANT MODIFY, SELECT ON TABLE fhir_bundle_zerobus TO `" ||spn_application_id || "`;";

SELECT grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE grnt_stmnt;

In [0]:
SHOW GRANTS ON TABLE fhir_bundle_zerobus;